# Desafio Pandas — Auditoria com Dependência entre Colunas

## Objetivo
Você recebeu um dataset de um marketplace. O problema é que várias colunas parecem corretas quando analisadas isoladamente, mas ficam incoerentes quando comparadas entre si.

Você deve limpar, padronizar, corrigir tipos, tratar nulos, remover duplicatas e criar colunas usando apenas os tópicos trabalhados em aula.

Use essas regras para encontrar inconsistências:

1. Se **status_pedido = pago**, então comprou deveria ser 1.
2. Se **status_pedido = cancelado**, então comprou deveria ser 0.
3. Se **comprou = 1**, então valor_carrinho deveria ser maior que 0 e qtd_itens maior que 0.
4. Se **qtd_itens = 0**, então valor_carrinho deveria ser 0.
5. Frete grátis só é esperado quando **valor_carrinho >= 200 ou uf = CE**.
6. **paginas_visitadas = 0** não faz sentido quando **tempo_site > 0**.
7. **tempo_site = 999** representa erro de captura.

## Setup do dataset
Execute a célula abaixo para criar o dataset bruto.

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv('./data/marketplace.csv')
df.head()

,Unnamed: 0,User ID,Idade Cliente,Genero,UF,Categoria Produto,Tempo_Site,Paginas_Visitadas,Valor_Carrinho,Frete,Usou Cupom,Comprou,Status Pedido,Qtd Itens
0,0,1860.0,18,M,PE,mercado,0.0,4,300.0,gratis,sim,cancelado,NaN,10.0
1,1,NaN,trinta,Masculino,sp,mercado,999.0,5,NaN,25.5,não,nao,NaN,NaN
2,2,2044.0,18,Feminino,sp,Roup@,999.0,100,quatrocentos,gratis,NaN,cancelado,NaN,4.0
3,3,1121.0,25,NaN,sp,mercado,0.0,3,NaN,R$ 30,NAO,0,NaN,2.0
4,4,1466.0,30,M,CE,Roup@,2.0,3,150,gratis,sim,1,NaN,0.0


## Problema 1 — Diagnóstico inicial

Investigue o dataset bruto. Responda:

- Quantas linhas e colunas existem?
- Quais são os nomes das colunas?
- Quais tipos parecem incorretos?
- Onde existem valores nulos?
- O **describe()** ajuda em todas as colunas? Por quê?

In [2]:
print(f'Dataset marketplace possui {df.shape[0]} linhas e {df.shape[1]} colunas')
print(f'Lista de nomes das colunas: {df.columns}')

Dataset marketplace possui 4680 linhas e 14 colunas
Lista de nomes das colunas: Index(['Unnamed: 0', 'User ID', 'Idade Cliente', 'Genero', 'UF',
       'Categoria Produto', 'Tempo_Site', 'Paginas_Visitadas',
       'Valor_Carrinho', 'Frete', 'Usou Cupom', 'Comprou', 'Status Pedido',
       'Qtd Itens'],
      dtype='str')


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4680 entries, 0 to 4679
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         4680 non-null   int64  
 1   User ID            4610 non-null   float64
 2   Idade Cliente      4272 non-null   str    
 3   Genero             3665 non-null   str    
 4   UF                 3671 non-null   str    
 5   Categoria Produto  4274 non-null   str    
 6   Tempo_Site         4151 non-null   float64
 7   Paginas_Visitadas  4680 non-null   int64  
 8   Valor_Carrinho     4274 non-null   str    
 9   Frete              4023 non-null   str    
 10  Usou Cupom         3535 non-null   str    
 11  Comprou            4680 non-null   str    
 12  Status Pedido      3166 non-null   str    
 13  Qtd Itens          4039 non-null   float64
dtypes: float64(3), int64(2), str(9)
memory usage: 652.2 KB


Unnamed: 0 não precisaria existir, pois é uma repetição dos índices, User ID poderia ser int, Valor_Carrinho e Frete deveriam ser float, Usou Cupom, Comprou poderia ser booleano e Qtd Itens poderia ser int.

In [4]:
df.isnull().sum()

Unnamed: 0              0
User ID                70
Idade Cliente         408
Genero               1015
UF                   1009
Categoria Produto     406
Tempo_Site            529
Paginas_Visitadas       0
Valor_Carrinho        406
Frete                 657
Usou Cupom           1145
Comprou                 0
Status Pedido        1514
Qtd Itens             641
dtype: int64

In [5]:
df.describe()

,Unnamed: 0,User ID,Tempo_Site,Paginas_Visitadas,Qtd Itens
count,4680.00000,4610.000000,4151.000000,4680.000000,4039.000000
mean,2339.50000,1557.078308,132.769453,14.275000,3.300322
std,1351.14396,318.649910,329.410637,30.207281,3.191460
min,0.00000,1000.000000,-5.000000,0.000000,0.000000
25%,1169.75000,1278.000000,0.000000,2.000000,1.000000
50%,2339.50000,1566.000000,10.000000,4.000000,2.000000
75%,3509.25000,1837.000000,25.000000,6.000000,4.000000
max,4679.00000,2099.000000,999.000000,100.000000,10.000000


Não ajuda em todas as colunas. Primeiramente porque é inútil em colunas como Unnamed:0 e User ID e segundamente porque algumas colunas que deveriam ter valores numéricos estão como strings.

## Problema 2 — Padronizar nomes das colunas

Renomeie as colunas para um padrão mais profissional:

- letras minúsculas
- sem espaços
- nomes consistentes

Exemplo: **User ID** para **user_id**.

In [6]:
clean_df = df.copy()

clean_df.drop(columns='Unnamed: 0', inplace=True)
clean_df.head()

,User ID,Idade Cliente,Genero,UF,Categoria Produto,Tempo_Site,Paginas_Visitadas,Valor_Carrinho,Frete,Usou Cupom,Comprou,Status Pedido,Qtd Itens
0,1860.0,18,M,PE,mercado,0.0,4,300.0,gratis,sim,cancelado,NaN,10.0
1,NaN,trinta,Masculino,sp,mercado,999.0,5,NaN,25.5,não,nao,NaN,NaN
2,2044.0,18,Feminino,sp,Roup@,999.0,100,quatrocentos,gratis,NaN,cancelado,NaN,4.0
3,1121.0,25,NaN,sp,mercado,0.0,3,NaN,R$ 30,NAO,0,NaN,2.0
4,1466.0,30,M,CE,Roup@,2.0,3,150,gratis,sim,1,NaN,0.0


In [7]:
def clean_str(x):
    return x.strip().replace(' ', '_').lower() 

clean_str(' User ID ')

'user_id'

In [8]:
new_columns = {}
for col in clean_df.columns.values:
    new_columns[col] = clean_str(col)

new_columns

{'User ID': 'user_id',
 'Idade Cliente': 'idade_cliente',
 'Genero': 'genero',
 'UF': 'uf',
 'Categoria Produto': 'categoria_produto',
 'Tempo_Site': 'tempo_site',
 'Paginas_Visitadas': 'paginas_visitadas',
 'Valor_Carrinho': 'valor_carrinho',
 'Frete': 'frete',
 'Usou Cupom': 'usou_cupom',
 'Comprou': 'comprou',
 'Status Pedido': 'status_pedido',
 'Qtd Itens': 'qtd_itens'}

In [9]:
clean_df.rename(columns=new_columns, inplace=True)
clean_df.head()

,user_id,idade_cliente,genero,uf,categoria_produto,tempo_site,paginas_visitadas,valor_carrinho,frete,usou_cupom,comprou,status_pedido,qtd_itens
0,1860.0,18,M,PE,mercado,0.0,4,300.0,gratis,sim,cancelado,NaN,10.0
1,NaN,trinta,Masculino,sp,mercado,999.0,5,NaN,25.5,não,nao,NaN,NaN
2,2044.0,18,Feminino,sp,Roup@,999.0,100,quatrocentos,gratis,NaN,cancelado,NaN,4.0
3,1121.0,25,NaN,sp,mercado,0.0,3,NaN,R$ 30,NAO,0,NaN,2.0
4,1466.0,30,M,CE,Roup@,2.0,3,150,gratis,sim,1,NaN,0.0


## Problema 3 — Detectar strings vazias

Atenção: string vazia **""** não aparece como nulo no **isnull()**.

Investigue colunas como **genero**, **uf**, **categoria_produto**, **usou_cupom** e **status_pedido**.

In [10]:
mask = clean_df.map(lambda x: pd.isna(x) or (isinstance(x, str) and x.strip() == ''))

mask.sum()

user_id                70
idade_cliente         408
genero               1015
uf                   1009
categoria_produto     406
tempo_site            529
paginas_visitadas       0
valor_carrinho        406
frete                 657
usou_cupom           1145
comprou                 0
status_pedido        1514
qtd_itens             641
dtype: int64

In [11]:
clean_df.replace(r'^\s*$', np.nan, regex=True, inplace=True).isna().sum()

user_id                70
idade_cliente         408
genero               1015
uf                   1009
categoria_produto     406
tempo_site            529
paginas_visitadas       0
valor_carrinho        406
frete                 657
usou_cupom           1145
comprou                 0
status_pedido        1514
qtd_itens             641
dtype: int64

## Problema 4 — Padronizar colunas de texto

Padronize:

- **genero**
- **uf**
- **categoria_produto**
- **usou_cupom**
- **status_pedido**

Use **str.strip()**, **str.lower()**, **str.upper()** e **replace()**.

In [12]:
clean_df['genero'].value_counts()

genero
m            554
f            543
Masculino    514
  M          504
Feminino     503
F            490
M            485
feminino      21
masculino     20
none          16
  m           15
Name: count, dtype: int64

As colunas **genero** e **usou_cupom** receberão o mesmo tratamento, primeira letra maiúscula representando o gênero (F e M) e uso do cupom (S ou N). A coluna gênero possui valor **none** como string.

In [13]:
clean_df['genero'] = clean_df['genero'].replace(r'(?i)^none$', np.nan, regex=True)

clean_df['genero'].value_counts()

genero
m            554
f            543
Masculino    514
  M          504
Feminino     503
F            490
M            485
feminino      21
masculino     20
  m           15
Name: count, dtype: int64

In [14]:
clean_df['genero'].str.strip().str[0].str.upper().value_counts()

genero
M    2092
F    1557
Name: count, dtype: int64

In [15]:
clean_df['usou_cupom'].str.strip().str[0].str.upper().value_counts()

usou_cupom
S    1798
N    1737
Name: count, dtype: int64

In [16]:
char_cols = ['genero', 'usou_cupom']

for col in char_cols:
    clean_df[col] = clean_df[col].str.strip().str[0].str.upper()

clean_df[char_cols].head()

,genero,usou_cupom
0,M,S
1,M,N
2,F,NaN
3,NaN,N
4,M,S


As demais colunas serão tratadas retirando os espaços em branco e ao por os valores para maiúsculos. **uf** deve ter valores **xx** e **none** substituídos para NaN e **categoria_produto** deve ter seus valores com @ substituídos por 'A'.

In [17]:
clean_df['uf'].str.strip().str.upper().value_counts()

uf
SP      1071
CE      1029
XX       528
PE       512
RJ       512
NONE      19
Name: count, dtype: int64

In [18]:
clean_df['categoria_produto'].str.strip().str.upper().value_counts()

categoria_produto
ELETRONICOS    1298
ROUPAS          849
MERCADO         840
CASA            819
ROUP@           468
Name: count, dtype: int64

In [19]:
clean_df['status_pedido'].str.strip().str.upper().value_counts()

status_pedido
PAGO         1636
CANCELADO     780
PENDENTE      750
Name: count, dtype: int64

In [20]:
upper_cols = ['uf', 'categoria_produto', 'status_pedido']

for col in upper_cols:
    clean_df[col] = clean_df[col].str.strip().str.upper()

clean_df[upper_cols].tail()

,uf,categoria_produto,status_pedido
4675,PE,CASA,PAGO
4676,CE,NaN,CANCELADO
4677,NaN,ROUP@,PAGO
4678,NONE,MERCADO,PENDENTE
4679,SP,ROUPAS,NaN


In [21]:
clean_df['uf'] = clean_df['uf'].replace(['NONE', 'XX'], np.nan)

clean_df['uf'].value_counts()

uf
SP    1071
CE    1029
PE     512
RJ     512
Name: count, dtype: int64

In [22]:
clean_df['categoria_produto'].str.replace('@', 'AS', regex=False).value_counts()

categoria_produto
ROUPAS         1317
ELETRONICOS    1298
MERCADO         840
CASA            819
Name: count, dtype: int64

In [23]:
clean_df['categoria_produto'] = clean_df['categoria_produto'].str.replace('@', 'AS', regex=False)

clean_df[upper_cols].tail()

,uf,categoria_produto,status_pedido
4675,PE,CASA,PAGO
4676,CE,NaN,CANCELADO
4677,NaN,ROUPAS,PAGO
4678,NaN,MERCADO,PENDENTE
4679,SP,ROUPAS,NaN


In [24]:
str_cols = upper_cols + char_cols

clean_df[str_cols].tail()

,uf,categoria_produto,status_pedido,genero,usou_cupom
4675,PE,CASA,PAGO,M,S
4676,CE,NaN,CANCELADO,F,N
4677,NaN,ROUPAS,PAGO,NaN,N
4678,NaN,MERCADO,PENDENTE,NaN,S
4679,SP,ROUPAS,NaN,M,NaN


## Problema 5 — Corrigir tipos numéricos

Corrija:

- **idade_cliente**
- **valor_carrinho**
- **frete**
- **qtd_itens**
- **comprou**

Cuidado com valores como **trinta**, **quatrocentos**, **R$ 400**, **R$400**, **gratis**, **sim**, **nao**, **cancelado**.

O primeiro tratamento será da coluna **comprou**. Para manter a consistência com **usou_cupom** a coluna será transformada em uma coluna categórica 'SIM', 'NAO' ou 'CANCELADO'.

In [25]:
clean_df['comprou'].astype(str).replace(['0', '1'], ['nao', 'sim']).str.upper().value_counts()

comprou
SIM          2059
NAO          1966
CANCELADO     655
Name: count, dtype: int64

In [26]:
clean_df['comprou'] = clean_df['comprou'].astype(str).replace(['0', '1'], ['nao', 'sim']).str.upper()

clean_df['comprou'].value_counts()

comprou
SIM          2059
NAO          1966
CANCELADO     655
Name: count, dtype: int64

O próximo tratamento será para as colunas que podem ser contadas com valores inteiros (**idade_cliente** e **qtd_itens**). No caso, apenas **idade_cliente** precisa ser tratado pois possui strings em seus valores.

In [27]:
clean_df['idade_cliente'].str.strip().str.replace('trinta', '30').value_counts()

idade_cliente
30    1265
35     830
55     459
18     445
22     440
42     432
25     401
Name: count, dtype: int64

In [28]:
clean_df['idade_cliente'] = clean_df['idade_cliente'].str.strip().str.replace('trinta', '30').astype(float)

clean_df['idade_cliente'].value_counts()

idade_cliente
30.0    1265
35.0     830
55.0     459
18.0     445
22.0     440
42.0     432
25.0     401
Name: count, dtype: int64

In [29]:
clean_df['qtd_itens'].value_counts()

qtd_itens
2.0     739
4.0     682
3.0     680
0.0     659
10.0    645
1.0     634
Name: count, dtype: int64

Agora para valores de ponto flutuante, que possui lógica praticamente idêntica (**valor_carrinho** e **frete**).

In [30]:
print(clean_df['valor_carrinho'].value_counts())

valor_carrinho
300             526
0               449
200             442
50              431
R$400           418
300.0           412
quatrocentos    409
R$ 400          409
150             400
100             378
Name: count, dtype: int64


In [31]:
clean_df['frete'] = clean_df['frete'].str.strip().replace(['R$ 30', 'gratis'], ['30', '0']).astype(float)

clean_df['frete'].value_counts()

frete
0.0     1305
25.5     727
30.0     704
10.0     656
20.0     631
Name: count, dtype: int64

In [32]:
clean_df['valor_carrinho'] = clean_df['valor_carrinho'].str.strip().replace(['R$400', 'R$ 400', 'quatrocentos'], '400').astype(float)

clean_df['valor_carrinho'].value_counts()

valor_carrinho
400.0    1236
300.0     938
0.0       449
200.0     442
50.0      431
150.0     400
100.0     378
Name: count, dtype: int64

In [33]:
num_cols = ['comprou', 'idade_cliente', 'qtd_itens', 'valor_carrinho', 'frete']

clean_df[num_cols].head()

,comprou,idade_cliente,qtd_itens,valor_carrinho,frete
0,CANCELADO,18.0,10.0,300.0,0.0
1,NAO,30.0,NaN,NaN,25.5
2,CANCELADO,18.0,4.0,400.0,0.0
3,NAO,25.0,2.0,NaN,30.0
4,SIM,30.0,0.0,150.0,0.0


In [34]:
clean_df.dtypes

user_id              float64
idade_cliente        float64
genero                   str
uf                       str
categoria_produto        str
tempo_site           float64
paginas_visitadas      int64
valor_carrinho       float64
frete                float64
usou_cupom               str
comprou                  str
status_pedido            str
qtd_itens            float64
dtype: object

In [35]:
clean_df.head(10)

,user_id,idade_cliente,genero,uf,categoria_produto,tempo_site,paginas_visitadas,valor_carrinho,frete,usou_cupom,comprou,status_pedido,qtd_itens
0,1860.0,18.0,M,PE,MERCADO,0.0,4,300.0,0.0,S,CANCELADO,NaN,10.0
1,NaN,30.0,M,SP,MERCADO,999.0,5,NaN,25.5,N,NAO,NaN,NaN
2,2044.0,18.0,F,SP,ROUPAS,999.0,100,400.0,0.0,NaN,CANCELADO,NaN,4.0
3,1121.0,25.0,NaN,SP,MERCADO,0.0,3,NaN,30.0,N,NAO,NaN,2.0
4,1466.0,30.0,M,CE,ROUPAS,2.0,3,150.0,0.0,S,SIM,NaN,0.0
5,1330.0,30.0,F,NaN,ELETRONICOS,2.0,5,400.0,0.0,N,SIM,NaN,2.0
6,1087.0,NaN,NaN,NaN,ELETRONICOS,10.0,6,150.0,0.0,NaN,SIM,PAGO,2.0
7,1871.0,30.0,NaN,NaN,ELETRONICOS,5.0,3,300.0,0.0,S,SIM,PAGO,NaN
8,1130.0,42.0,M,NaN,ROUPAS,2.0,1,0.0,10.0,N,NAO,NaN,0.0
9,1769.0,30.0,F,SP,ROUPAS,2.0,0,400.0,10.0,N,SIM,PAGO,0.0


## Problema 6 — Tratar valores inválidos de navegação

Trate:

- **tempo_site = -5**
- **tempo_site = 0**
- **tempo_site = 999**
- **paginas_visitadas = 0**
- **paginas_visitadas = 100**

Justifique suas escolhas.

O valor de **paginas_visitadas** 100 serão transformados em valores NaN pois será considerado um erro de captação. Transformar em NaN irá explicitar esse erro.

In [37]:
clean_df['paginas_visitadas'].value_counts()

paginas_visitadas
5      567
8      557
4      546
100    514
0      512
3      508
1      496
2      492
6      488
Name: count, dtype: int64

In [38]:
clean_df[clean_df['paginas_visitadas'] == 100].head()

,user_id,idade_cliente,genero,uf,categoria_produto,tempo_site,paginas_visitadas,valor_carrinho,frete,usou_cupom,comprou,status_pedido,qtd_itens
2,2044.0,18.0,F,SP,ROUPAS,999.0,100,400.0,0.0,NaN,CANCELADO,NaN,4.0
16,1021.0,42.0,F,NaN,MERCADO,999.0,100,400.0,NaN,N,NAO,NaN,1.0
21,2082.0,35.0,M,CE,CASA,10.0,100,300.0,0.0,N,NAO,PAGO,0.0
31,1646.0,NaN,M,CE,MERCADO,999.0,100,200.0,10.0,S,SIM,PAGO,1.0
49,2025.0,25.0,NaN,CE,ROUPAS,-5.0,100,300.0,0.0,N,SIM,PENDENTE,10.0


In [39]:
clean_df.loc[clean_df['paginas_visitadas'] == 100, 'paginas_visitadas'] = np.nan

clean_df['paginas_visitadas'].value_counts()

paginas_visitadas
5.0    567
8.0    557
4.0    546
0.0    512
3.0    508
1.0    496
2.0    492
6.0    488
Name: count, dtype: int64

De modo semelhante, os valores 999 e -5 encontrados em **tempo_site** serão transformados em NaN para explicitar esse erro de captação.

In [50]:
clean_df[ (clean_df['tempo_site'] == 999) | (clean_df['tempo_site'] < 0.0) ].tail()

,user_id,idade_cliente,genero,uf,categoria_produto,tempo_site,paginas_visitadas,valor_carrinho,frete,usou_cupom,comprou,status_pedido,qtd_itens
4661,1309.0,35.0,F,SP,ELETRONICOS,-5.0,5.0,300.0,0.0,S,NAO,NaN,2.0
4663,1039.0,35.0,F,CE,ROUPAS,999.0,8.0,400.0,NaN,S,NAO,PAGO,10.0
4676,1905.0,22.0,F,CE,NaN,-5.0,4.0,0.0,10.0,N,SIM,CANCELADO,10.0
4677,1082.0,22.0,NaN,NaN,ROUPAS,999.0,4.0,0.0,10.0,N,CANCELADO,PAGO,10.0
4679,2045.0,35.0,M,SP,ROUPAS,-5.0,2.0,300.0,NaN,NaN,NAO,NaN,4.0


In [52]:
clean_df['tempo_site'].replace([999, -5], np.nan).value_counts()

tempo_site
25.0    559
10.0    548
0.0     546
15.0    494
5.0     486
2.0     475
Name: count, dtype: int64

In [53]:
clean_df['tempo_site'] = clean_df['tempo_site'].replace([999, -5], np.nan)

clean_df.tail()

,user_id,idade_cliente,genero,uf,categoria_produto,tempo_site,paginas_visitadas,valor_carrinho,frete,usou_cupom,comprou,status_pedido,qtd_itens
4675,1444.0,35.0,M,PE,CASA,5.0,1.0,300.0,10.0,S,SIM,PAGO,1.0
4676,1905.0,22.0,F,CE,NaN,NaN,4.0,0.0,10.0,N,SIM,CANCELADO,10.0
4677,1082.0,22.0,NaN,NaN,ROUPAS,NaN,4.0,0.0,10.0,N,CANCELADO,PAGO,10.0
4678,1592.0,30.0,NaN,NaN,MERCADO,10.0,8.0,150.0,25.5,S,SIM,PENDENTE,0.0
4679,2045.0,35.0,M,SP,ROUPAS,NaN,2.0,300.0,NaN,NaN,NAO,NaN,4.0


## Problema 7 — Dependência 1: compra e status do pedido

Encontre e corrija incoerências:

- **status_pedido = pago** mas **comprou = 0**
- **status_pedido = cancelado** mas **comprou = 1**

Explique qual coluna você escolheu como mais confiável.

In [67]:
status_pago = (clean_df['status_pedido'] == 'PAGO') & (clean_df['comprou'] == 'NAO')

print(f'Número de pedidos com status pago e comprou como NAO: {len( clean_df[status_pago] )}')
clean_df[status_pago].head()

Número de pedidos com status pago e comprou como NAO: 733


,user_id,idade_cliente,genero,uf,categoria_produto,tempo_site,paginas_visitadas,valor_carrinho,frete,usou_cupom,comprou,status_pedido,qtd_itens
10,1343.0,55.0,F,SP,NaN,0.0,1.0,400.0,20.0,S,NAO,PAGO,NaN
15,1459.0,30.0,M,RJ,ELETRONICOS,25.0,4.0,400.0,0.0,S,NAO,PAGO,10.0
21,2082.0,35.0,M,CE,CASA,10.0,NaN,300.0,0.0,N,NAO,PAGO,0.0
25,1189.0,42.0,M,SP,ELETRONICOS,NaN,2.0,100.0,0.0,S,NAO,PAGO,3.0
50,2021.0,18.0,F,PE,CASA,0.0,NaN,200.0,10.0,S,NAO,PAGO,0.0


In [ ]:
status_cancel = (clean_df['status_pedido'] == 'CANCELADO') & (clean_df['comprou'] == 'SIM')

print(f'Número de pedidos com status CANCELADO e comprou como SIM: {len( clean_df[status_cancel] )}')
clean_df[status_cancel].head()

Número de pedidos com status CANCELADO e comprou como SIM: 367


,user_id,idade_cliente,genero,uf,categoria_produto,tempo_site,paginas_visitadas,valor_carrinho,frete,usou_cupom,comprou,status_pedido,qtd_itens
12,1385.0,55.0,M,NaN,ELETRONICOS,NaN,3.0,300.0,10.0,NaN,SIM,CANCELADO,3.0
29,1562.0,42.0,M,NaN,MERCADO,NaN,6.0,150.0,0.0,S,SIM,CANCELADO,NaN
38,1013.0,55.0,M,SP,ROUPAS,5.0,6.0,400.0,20.0,S,SIM,CANCELADO,2.0
44,1955.0,35.0,F,NaN,ELETRONICOS,2.0,4.0,300.0,10.0,NaN,SIM,CANCELADO,2.0
62,1295.0,30.0,M,NaN,ROUPAS,10.0,0.0,300.0,30.0,S,SIM,CANCELADO,2.0


In [60]:
clean_df['status_pedido'].value_counts()

status_pedido
PAGO         1636
CANCELADO     780
PENDENTE      750
Name: count, dtype: int64

In [61]:
clean_df['comprou'].value_counts()

comprou
SIM          2059
NAO          1966
CANCELADO     655
Name: count, dtype: int64

A coluna que será considerada mais confiável no tratamento dos dados é a coluna **status_pedido** pois será assumido que a informação de confirmação do pagamento está relacionada a uma API de *netbanking*, criando assim uma coerência com informações do banco e do cliente. para as colunas com estado 'PAGO' mas comprou 'NAO', essas serão transformadas em 'SIM', e colunas com estado 'CANCELADO' e comprou 'NAO' terá seu estado modificado para 'CANCELADO'.

In [70]:
clean_df.loc[ status_pago, 'comprou' ] = 'SIM'
clean_df.loc[ status_cancel, 'comprou' ] = 'CANCELADO'

clean_df['comprou'].value_counts()

comprou
SIM          2425
NAO          1233
CANCELADO    1022
Name: count, dtype: int64

## Problema 8 — Dependência 2: compra, valor e quantidade

Encontre registros incoerentes:

- **comprou = 1** com **valor_carrinho = 0**
- **comprou = 1** com **qtd_itens = 0**
- **qtd_itens = 0** com **valor_carrinho > 0**

Corrija ou remova, justificando.

In [ ]:
carrinho_zero = (clean_df['comprou'] == 'SIM') & (clean_df['valor_carrinho'] == 0)

print('Número de entradas com comprou SIM e valor_carrinho ZERO:', len( clean_df[carrinho_zero] ))
clean_df[carrinho_zero].head()

Número de entradas com comprou SIM e valor_carrinho ZERO: 242


,user_id,idade_cliente,genero,uf,categoria_produto,tempo_site,paginas_visitadas,valor_carrinho,frete,usou_cupom,comprou,status_pedido,qtd_itens
17,1252.0,NaN,M,NaN,CASA,15.0,6.0,0.0,0.0,N,SIM,PAGO,3.0
40,1776.0,22.0,M,SP,ELETRONICOS,NaN,4.0,0.0,10.0,S,SIM,PAGO,3.0
77,1647.0,22.0,M,PE,ROUPAS,5.0,3.0,0.0,25.5,S,SIM,PENDENTE,NaN
101,1612.0,22.0,NaN,NaN,ELETRONICOS,NaN,4.0,0.0,0.0,S,SIM,PENDENTE,NaN
117,1095.0,35.0,NaN,SP,ROUPAS,NaN,4.0,0.0,20.0,S,SIM,NaN,2.0


In [77]:
qtd_zero = (clean_df['qtd_itens'] == 0) & (clean_df['comprou'] == 'SIM')

print('Número de entradas com comprou SIM e qtd_itens ZERO:', len( clean_df[qtd_zero] ))
clean_df[qtd_zero].head()

Número de entradas com comprou SIM e qtd_itens ZERO: 339


,user_id,idade_cliente,genero,uf,categoria_produto,tempo_site,paginas_visitadas,valor_carrinho,frete,usou_cupom,comprou,status_pedido,qtd_itens
4,1466.0,30.0,M,CE,ROUPAS,2.0,3.0,150.0,0.0,S,SIM,NaN,0.0
9,1769.0,30.0,F,SP,ROUPAS,2.0,0.0,400.0,10.0,N,SIM,PAGO,0.0
21,2082.0,35.0,M,CE,CASA,10.0,NaN,300.0,0.0,N,SIM,PAGO,0.0
27,1686.0,35.0,NaN,SP,ELETRONICOS,25.0,3.0,400.0,NaN,S,SIM,PAGO,0.0
32,1020.0,18.0,M,CE,ROUPAS,25.0,6.0,300.0,10.0,S,SIM,PAGO,0.0


Como **comprou** já foi compatibilizada com **status_pedido** ela será a referência para as ações. Em caso de **comprou** como 'SIM', **qtd_itens** ou **valor_carrinho** 0 serão colocados como NaN e encarados como erro de captação dos valores.

In [81]:
clean_df.loc[carrinho_zero, 'valor_carrinho'] = np.nan

carrinho_zero = (clean_df['comprou'] == 'SIM') & (clean_df['valor_carrinho'] == 0)
print('Número de entradas com comprou SIM e valor_carrinho ZERO:', len( clean_df[carrinho_zero] ))

Número de entradas com comprou SIM e valor_carrinho ZERO: 0


In [82]:
clean_df.loc[qtd_zero, 'qtd_itens'] = np.nan

qtd_zero = (clean_df['qtd_itens'] == 0) & (clean_df['comprou'] == 'SIM')
print('Número de entradas com comprou SIM e qtd_itens ZERO:', len( clean_df[qtd_zero] ))

Número de entradas com comprou SIM e qtd_itens ZERO: 0


No último caso o valor considerado correto será o pago em **valor_carrinho**, novamente para manter coerência com a informação em **comprou** e possíveis comunicações com pagamentos em ambientes externos. O valor em **qtd_itens** será transformado em NaN para explicitar erro de captação.

In [83]:
carrinho_qtd = (clean_df['valor_carrinho'] > 0) & (clean_df['qtd_itens'] == 0)

print('Número de entradas com valor_carrinho MAIOR QUE ZERO e qtd_itens ZERO:', len( clean_df[carrinho_qtd] ))
clean_df[carrinho_qtd].head()

Número de entradas com valor_carrinho MAIOR QUE ZERO e qtd_itens ZERO: 265


,user_id,idade_cliente,genero,uf,categoria_produto,tempo_site,paginas_visitadas,valor_carrinho,frete,usou_cupom,comprou,status_pedido,qtd_itens
45,1508.0,42.0,M,NaN,ELETRONICOS,2.0,2.0,300.0,30.0,N,CANCELADO,PENDENTE,0.0
88,1134.0,NaN,M,RJ,CASA,5.0,NaN,50.0,25.5,S,CANCELADO,CANCELADO,0.0
111,2038.0,30.0,M,CE,NaN,15.0,8.0,400.0,10.0,NaN,NAO,CANCELADO,0.0
142,1951.0,30.0,F,NaN,ELETRONICOS,25.0,0.0,300.0,NaN,NaN,CANCELADO,PAGO,0.0
161,1197.0,22.0,M,NaN,NaN,25.0,8.0,300.0,0.0,S,CANCELADO,PAGO,0.0


In [84]:
clean_df.loc[ carrinho_qtd, 'qtd_itens' ] = np.nan

carrinho_qtd = (clean_df['valor_carrinho'] > 0) & (clean_df['qtd_itens'] == 0)
print('Número de entradas com valor_carrinho MAIOR QUE ZERO e qtd_itens ZERO:', len( clean_df[carrinho_qtd] ))

Número de entradas com valor_carrinho MAIOR QUE ZERO e qtd_itens ZERO: 0


In [85]:
clean_df.head(10)

,user_id,idade_cliente,genero,uf,categoria_produto,tempo_site,paginas_visitadas,valor_carrinho,frete,usou_cupom,comprou,status_pedido,qtd_itens
0,1860.0,18.0,M,PE,MERCADO,0.0,4.0,300.0,0.0,S,CANCELADO,NaN,10.0
1,NaN,30.0,M,SP,MERCADO,NaN,5.0,NaN,25.5,N,NAO,NaN,NaN
2,2044.0,18.0,F,SP,ROUPAS,NaN,NaN,400.0,0.0,NaN,CANCELADO,NaN,4.0
3,1121.0,25.0,NaN,SP,MERCADO,0.0,3.0,NaN,30.0,N,NAO,NaN,2.0
4,1466.0,30.0,M,CE,ROUPAS,2.0,3.0,150.0,0.0,S,SIM,NaN,NaN
5,1330.0,30.0,F,NaN,ELETRONICOS,2.0,5.0,400.0,0.0,N,SIM,NaN,2.0
6,1087.0,NaN,NaN,NaN,ELETRONICOS,10.0,6.0,150.0,0.0,NaN,SIM,PAGO,2.0
7,1871.0,30.0,NaN,NaN,ELETRONICOS,5.0,3.0,300.0,0.0,S,SIM,PAGO,NaN
8,1130.0,42.0,M,NaN,ROUPAS,2.0,1.0,0.0,10.0,N,NAO,NaN,0.0
9,1769.0,30.0,F,SP,ROUPAS,2.0,0.0,400.0,10.0,N,SIM,PAGO,NaN


## Problema 9 — Dependência 3: frete grátis suspeito

Pela regra de negócio, frete grátis só é esperado quando:

- **valor_carrinho >= 200**, ou
- **uf = CE**

Encontre registros suspeitos com **frete = 0**, **valor_carrinho < 200** e **uf != CE**.

In [86]:
frete_gratis = (clean_df['valor_carrinho'] >= 200) | (clean_df['uf'] == 'CE')

print(f'Número de compras elegíveis para frete grátis: {len( clean_df[frete_gratis] )}')

Número de compras elegíveis para frete grátis: 3049


In [91]:
frete_suspeito = (~frete_gratis) & (clean_df['frete'] == 0)

print(f'Número de compras com frete suspeito: {len (clean_df[frete_suspeito]) }')
clean_df[frete_suspeito].head()

Número de compras com frete suspeito: 510


,user_id,idade_cliente,genero,uf,categoria_produto,tempo_site,paginas_visitadas,valor_carrinho,frete,usou_cupom,comprou,status_pedido,qtd_itens
6,1087.0,NaN,NaN,NaN,ELETRONICOS,10.0,6.0,150.0,0.0,NaN,SIM,PAGO,2.0
17,1252.0,NaN,M,NaN,CASA,15.0,6.0,NaN,0.0,N,SIM,PAGO,3.0
19,1856.0,35.0,M,NaN,ROUPAS,NaN,8.0,100.0,0.0,NaN,NAO,CANCELADO,10.0
25,1189.0,42.0,M,SP,ELETRONICOS,NaN,2.0,100.0,0.0,S,SIM,PAGO,3.0
29,1562.0,42.0,M,NaN,MERCADO,NaN,6.0,150.0,0.0,S,CANCELADO,CANCELADO,NaN


## Problema 10 — Remover duplicatas no momento certo

Remova duplicatas somente depois da padronização.

Compare a quantidade de duplicatas antes e depois da limpeza.

In [100]:
print('Número de linhas no Dataset completo:', len(clean_df))

print('\nA linha é duplicada?')
clean_df.duplicated().value_counts()

Número de linhas no Dataset completo: 4680

A linha é duplicada?


False    4503
True      177
Name: count, dtype: int64

In [101]:
clean_df.drop_duplicates(inplace=True)

print('Número de linhas após limpeza de duplicados:', len(clean_df))

Número de linhas após limpeza de duplicados: 4503


## Problema 11 — Seleção, filtros e ordenação

Responda com filtros e ordenações:

1. Quais são os 10 maiores carrinhos?
2. Quais clientes compraram e usaram cupom?
3. Quais registros têm categoria **eletronicos** e carrinho acima de 200?
4. Mostre linhas e colunas específicas com **iloc()**.

In [115]:
top_carts = clean_df['valor_carrinho'].dropna().sort_values(ascending=False).head(10).index

print('Maiores 10 carrinhos:')
clean_df.loc[top_carts]

Maiores 10 carrinhos:


,user_id,idade_cliente,genero,uf,categoria_produto,tempo_site,paginas_visitadas,valor_carrinho,frete,usou_cupom,comprou,status_pedido,qtd_itens
13,1955.0,30.0,M,RJ,CASA,10.0,6.0,400.0,10.0,N,CANCELADO,NaN,2.0
15,1459.0,30.0,M,RJ,ELETRONICOS,25.0,4.0,400.0,0.0,S,SIM,PAGO,10.0
16,1021.0,42.0,F,NaN,MERCADO,NaN,NaN,400.0,NaN,N,NAO,NaN,1.0
2,2044.0,18.0,F,SP,ROUPAS,NaN,NaN,400.0,0.0,NaN,CANCELADO,NaN,4.0
23,1699.0,35.0,NaN,NaN,MERCADO,2.0,0.0,400.0,10.0,N,NAO,PENDENTE,NaN
5,1330.0,30.0,F,NaN,ELETRONICOS,2.0,5.0,400.0,0.0,N,SIM,NaN,2.0
4518,NaN,30.0,NaN,CE,CASA,10.0,0.0,400.0,25.5,S,SIM,PENDENTE,NaN
2406,1513.0,22.0,M,NaN,ROUPAS,NaN,0.0,400.0,0.0,S,SIM,PAGO,1.0
2408,2039.0,25.0,M,CE,ELETRONICOS,25.0,4.0,400.0,0.0,NaN,SIM,PAGO,1.0
2411,1880.0,25.0,NaN,CE,CASA,15.0,6.0,400.0,20.0,NaN,SIM,PENDENTE,3.0


In [117]:
clientes_cupom = (clean_df['comprou'] == 'SIM') & (clean_df['usou_cupom'] == 'S')

print('Número de clientes que compraram com cupom:', len(clean_df[clientes_cupom]) )
clean_df[clientes_cupom].head(10)

Número de clientes que compraram com cupom: 944


,user_id,idade_cliente,genero,uf,categoria_produto,tempo_site,paginas_visitadas,valor_carrinho,frete,usou_cupom,comprou,status_pedido,qtd_itens
4,1466.0,30.0,M,CE,ROUPAS,2.0,3.0,150.0,0.0,S,SIM,NaN,NaN
7,1871.0,30.0,NaN,NaN,ELETRONICOS,5.0,3.0,300.0,0.0,S,SIM,PAGO,NaN
10,1343.0,55.0,F,SP,NaN,0.0,1.0,400.0,20.0,S,SIM,PAGO,NaN
15,1459.0,30.0,M,RJ,ELETRONICOS,25.0,4.0,400.0,0.0,S,SIM,PAGO,10.0
25,1189.0,42.0,M,SP,ELETRONICOS,NaN,2.0,100.0,0.0,S,SIM,PAGO,3.0
27,1686.0,35.0,NaN,SP,ELETRONICOS,25.0,3.0,400.0,NaN,S,SIM,PAGO,NaN
31,1646.0,NaN,M,CE,MERCADO,NaN,NaN,200.0,10.0,S,SIM,PAGO,1.0
32,1020.0,18.0,M,CE,ROUPAS,25.0,6.0,300.0,10.0,S,SIM,PAGO,NaN
33,1840.0,55.0,NaN,PE,CASA,25.0,5.0,200.0,0.0,S,SIM,PAGO,1.0
35,1387.0,NaN,F,NaN,ROUPAS,25.0,1.0,50.0,25.5,S,SIM,PAGO,3.0


In [118]:
eletronicos_carrinho = (clean_df['categoria_produto'] == 'ELETRONICOS') & (clean_df['valor_carrinho'] >= 200)

print('Número de pedidos com categoria ELETRONICOS e valor_carrinho MAIOR QUE 200:', len(clean_df[eletronicos_carrinho]) )
clean_df[eletronicos_carrinho].head(10)

Número de pedidos com categoria ELETRONICOS e valor_carrinho MAIOR QUE 200: 698


,user_id,idade_cliente,genero,uf,categoria_produto,tempo_site,paginas_visitadas,valor_carrinho,frete,usou_cupom,comprou,status_pedido,qtd_itens
5,1330.0,30.0,F,NaN,ELETRONICOS,2.0,5.0,400.0,0.0,N,SIM,NaN,2.0
7,1871.0,30.0,NaN,NaN,ELETRONICOS,5.0,3.0,300.0,0.0,S,SIM,PAGO,NaN
12,1385.0,55.0,M,NaN,ELETRONICOS,NaN,3.0,300.0,10.0,NaN,CANCELADO,CANCELADO,3.0
15,1459.0,30.0,M,RJ,ELETRONICOS,25.0,4.0,400.0,0.0,S,SIM,PAGO,10.0
20,1474.0,35.0,NaN,CE,ELETRONICOS,NaN,3.0,300.0,NaN,N,SIM,NaN,NaN
27,1686.0,35.0,NaN,SP,ELETRONICOS,25.0,3.0,400.0,NaN,S,SIM,PAGO,NaN
28,1957.0,30.0,M,NaN,ELETRONICOS,15.0,1.0,400.0,10.0,S,NAO,NaN,3.0
44,1955.0,35.0,F,NaN,ELETRONICOS,2.0,4.0,300.0,10.0,NaN,CANCELADO,CANCELADO,2.0
45,1508.0,42.0,M,NaN,ELETRONICOS,2.0,2.0,300.0,30.0,N,CANCELADO,PENDENTE,NaN
70,1187.0,22.0,NaN,NaN,ELETRONICOS,0.0,0.0,400.0,30.0,S,SIM,PAGO,2.0


In [121]:
clean_df.iloc[:10, [1, 7, -3]]

,idade_cliente,valor_carrinho,comprou
0,18.0,300.0,CANCELADO
1,30.0,NaN,NAO
2,18.0,400.0,CANCELADO
3,25.0,NaN,NAO
4,30.0,150.0,SIM
5,30.0,400.0,SIM
6,NaN,150.0,SIM
7,30.0,300.0,SIM
8,42.0,0.0,NAO
9,30.0,400.0,SIM


## Problema 12 — Criação de colunas

Crie:

- **ticket_por_item = valor_carrinho / qtd_itens**
- **valor_total = valor_carrinho + frete**
- **usuario_valioso = 1** quando **comprou = 1**, **valor_total >= 250** e **tempo_site >= 10**; caso contrário, **0**

Cuidado com divisão por zero.

In [131]:
clean_df['ticket_por_item'] = (clean_df['valor_carrinho'] / clean_df['qtd_itens']).round(2)

clean_df[['valor_carrinho', 'qtd_itens', 'ticket_por_item']].head()

,valor_carrinho,qtd_itens,ticket_por_item
0,300.0,10.0,30.0
1,NaN,NaN,NaN
2,400.0,4.0,100.0
3,NaN,2.0,NaN
4,150.0,NaN,NaN


In [132]:
clean_df['valor_total'] = clean_df['valor_carrinho'] + clean_df['frete']

clean_df[['valor_carrinho', 'frete', 'valor_total']].head()

,valor_carrinho,frete,valor_total
0,300.0,0.0,300.0
1,NaN,25.5,NaN
2,400.0,0.0,400.0
3,NaN,30.0,NaN
4,150.0,0.0,150.0


In [135]:
usuario_valioso = (clean_df['comprou'] == 'SIM') & (clean_df['valor_total'] >= 250) & (clean_df['tempo_site'] >= 10)

print('Número de usuários valiosos:', len( clean_df[usuario_valioso] ))
clean_df[usuario_valioso].head()

Número de usuários valiosos: 303


,user_id,idade_cliente,genero,uf,categoria_produto,tempo_site,paginas_visitadas,valor_carrinho,frete,usou_cupom,comprou,status_pedido,qtd_itens,ticket_por_item,valor_total
15,1459.0,30.0,M,RJ,ELETRONICOS,25.0,4.0,400.0,0.0,S,SIM,PAGO,10.0,40.0,400.0
21,2082.0,35.0,M,CE,CASA,10.0,NaN,300.0,0.0,N,SIM,PAGO,NaN,NaN,300.0
32,1020.0,18.0,M,CE,ROUPAS,25.0,6.0,300.0,10.0,S,SIM,PAGO,NaN,NaN,310.0
60,1455.0,42.0,M,NaN,MERCADO,25.0,5.0,300.0,30.0,N,SIM,PENDENTE,3.0,100.0,330.0
65,1878.0,35.0,M,NaN,MERCADO,10.0,2.0,300.0,20.0,S,SIM,NaN,10.0,30.0,320.0


In [ ]:
clean_df['usuario_valioso'] = 0
clean_df.loc[usuario_valioso, 'usuario_valioso'] = 1

clean_df.loc[ 15:25, ['comprou', 'valor_total', 'tempo_site', 'usuario_valioso']]

,comprou,valor_total,tempo_site,usuario_valioso
15,SIM,400.0,25.0,1
16,NAO,NaN,NaN,0
17,SIM,NaN,15.0,0
18,NAO,30.0,5.0,0
19,NAO,100.0,NaN,0
20,SIM,NaN,NaN,0
21,SIM,300.0,10.0,1
22,CANCELADO,200.0,NaN,0
23,NAO,410.0,2.0,0
24,SIM,300.0,NaN,0
